# Notebook 16 — HARDREP-001: GPU-trained confusion-aware representation IDS

**Different from all prior CARES correction experiments:** This is a neural network trained end-to-end on genuine network flow features. It learns a **residual embedding** using class-balanced supervised learning, in-batch supervised contrastive learning, and a margin penalty applied to the observed hard pairs **DoS/Exploits, Backdoor/Exploits, Analysis/Exploits**.

**What is tested:** Genuine multiclass ten-class macro-F1, accuracy, per-class recalls and full confusion matrices for the trained model, both old frozen LightGBM baselines, and two fixed soft-probability ensembles. Nothing is fabricated or benchmarked on the unseen official test. A high score is **not guaranteed** and novelty needs relevant prior-work differentiation.

**Instructions:** Create a Kaggle notebook from this file, attach `dsshayan/unsw-nb15`, turn Internet ON, authorize existing `pushRare` Kaggle secret, **turn GPU ON (T4 or P100)**. Use `Save Version → Save & Run All` for cloud execution. One notebook cell performs preprocessing, training with every-two-epoch live logs, best checkpoint saving, validation and numerical CSV/NPZ export. The checkpoint and raw predictions are kept **only in Kaggle output**, not published to GitHub. Once the run completes, we will archive the private model as a verified draft GitHub Release, then sync numeric plots and metrics. Keep the successful Kaggle version outputs; do not repeat training until they are secured.

**Protocol:** Uses only original 140,328 development-training target rows, 35,005 original validation targets, and 42 raw flow columns; official UNSW-NB15 test remains untouched. It optimizes an explicitly identified internal validation score (not an unbiased test metric).

In [ ]:
# One cell: authentic GPU architecture training on frozen SCV development targets.
from pathlib import Path
import hashlib,os,subprocess,sys
from kaggle_secrets import UserSecretsClient
REPO=Path("/kaggle/working/rare-attack-ids-hardrep")
BASE=Path("/kaggle/working/research_outputs")
RUN=BASE/"HARDREP-001-S42"
CSV=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
assert CSV.is_file(),"Attach dataset dsshayan/unsw-nb15"
with CSV.open("rb") as f:
 assert hashlib.file_digest(f,"sha256").hexdigest()=="bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
token=UserSecretsClient().get_secret("pushRare")
assert token,"Enable Kaggle secret pushRare for private frozen baseline recovery"
if not REPO.exists():
 subprocess.run(["git","clone","--depth","1","https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git",str(REPO)],check=True)
else:
 subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
import torch
print("TORCH:",torch.__version__,"GPU AVAILABLE:",torch.cuda.is_available(),flush=True)
assert torch.cuda.is_available(),"Notebook 16 requires a Kaggle GPU (T4 or P100); enable GPU accelerator."
env=os.environ.copy()
env["GITHUB_TOKEN"]=token
env["PYTHONPATH"]=str(REPO/"src")
env["OMP_NUM_THREADS"]="4"
env["OPENBLAS_NUM_THREADS"]="1"
for script,archive in [("restore_scv_release.py","SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar"),("restore_ctrl003_tabular.py","CTRL-TAB-001-S42_FULL_ARTIFACTS.tar")]:
 subprocess.run([sys.executable,str(REPO/"scripts"/script),"--base",str(BASE),
                 "--archive",str(Path("/kaggle/working")/archive)],
                cwd=str(REPO),env=env,check=True)
subprocess.run([sys.executable,str(REPO/"scripts/run_hardrep001_gpu.py"),
 "--repo",str(REPO),"--reference",str(BASE/"SCV-IDS-REPRO-001"),
 "--baseline",str(BASE/"CTRL-TAB-001-S42"),
 "--config",str(REPO/"configs/hardrep001_gpu_seed42.json"),
 "--output",str(RUN)],cwd=str(REPO),env=env,check=True)
print("HARDREP TRAINED RESULTS:\n", (RUN/"variant_comparison.csv").read_text())
print("Private fitted checkpoint and authentic validation outputs saved at",RUN)